# 02 · Thinking in N dimensions / Pensar en N dimensiones

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/02-thinking-in-n-dimensions.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2">PART II · DEMO + EXERCISE · 20 MIN</span>

The goal of this notebook is simple:

**Do not read a tensor as a list of numbers. Read every axis as a question: “What does this axis count?”**

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 .7em">El objetivo de este cuaderno es sencillo:</div><div style="margin:0 0 0"><b>No leas un tensor como una lista de números. Lee cada eje como una pregunta: “¿Qué cuenta este eje?”</b></div></div>

## What you will be able to do / Lo que podrás hacer

- Read real image and video tensor shapes and explain every axis in plain language.
- Distinguish a **batch axis** from a **time axis**, even when the shapes are identical.
- See why shuffling independent examples can be acceptable while shuffling time changes the meaning.
- Build a padded order-5 video batch and use a validity mask to distinguish real frames from padding.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">Leer formas de tensores reales de imágenes y video y explicar cada eje en lenguaje sencillo.</li><li style="margin:.35em 0">Distinguir un <b>eje de lote</b> de un <b>eje temporal</b>, incluso cuando las formas son idénticas.</li><li style="margin:.35em 0">Entender por qué reorganizar ejemplos independientes puede ser válido mientras reorganizar el tiempo cambia el significado.</li><li style="margin:.35em 0">Construir un lote de video de orden 5 con padding y usar una máscara de validez para distinguir fotogramas reales de relleno.</li></ul></div>

## The five axis letters / Las cinco letras de ejes

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

| Letter / Letra | English | Español | The question it answers / La pregunta que responde |
|---|---|---|---|
| `N` | batch / examples | lote / ejemplos | How many independent examples? / ¿Cuántos ejemplos independientes? |
| `T` | time | tiempo | How many measured moments? / ¿Cuántos momentos medidos? |
| `H` | height | alto | How many pixel rows? / ¿Cuántas filas de píxeles? |
| `W` | width | ancho | How many pixel columns? / ¿Cuántas columnas de píxeles? |
| `C` | channels | canales | How many colour or measurement channels? / ¿Cuántos canales? |

Read <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(8,145,178,.14);border:1px solid rgba(8,145,178,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(N, T, H, W, C)</span> as a sentence and it stops being cryptic:
**examples × time × height × width × channels**.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Lee <code>(N, T, H, W, C)</code> como una frase — <b>ejemplos × tiempo × alto × ancho × canales</b> — y deja de ser críptica.</div>

## Setup / Preparación

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Three real datasets: handwritten digits, one RGB photograph, and a CC0 storm
video.

The video is downloaded once and checked with SHA-256, so we know exactly which
file we have.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Tres conjuntos reales: dígitos manuscritos, una fotografía RGB y un video de tormenta CC0. El video se descarga una vez y se verifica con SHA-256, así sabemos exactamente qué archivo tenemos.</div>

In [ ]:
%pip install -q "imageio[ffmpeg]"

import hashlib
import io
import urllib.request

import imageio.v3 as iio
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from sklearn.datasets import load_digits
from skimage import data

# Enable widgets in Google Colab when available.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

rng = np.random.default_rng(0)

# ---------------------------------------------------------------------------
# Real image data
# ---------------------------------------------------------------------------
digits = load_digits()
digit_batch = digits.images[:8].astype(np.float32)   # (N, H, W)
digit_labels = digits.target[:8]
real_digit = digit_batch[0]
real_photo = data.astronaut()                        # (H, W, C)

# ---------------------------------------------------------------------------
# Real video data: "Tormenta en l'Almadrava" by Nicolas Vigier, CC0
# ---------------------------------------------------------------------------
VIDEO_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/1/1e/"
    "Tormenta_en_l%27Almadrava.webm"
)
VIDEO_SHA256 = "e377fcdd2c79b55bce13c2c24b5dd7e412af39cd400eec548a79d0e59d79dc1b"
UA = "tensors-workshop/1.0 (https://github.com/project-delphi/tensors-workshop)"

def fetch_verified_video(url, expected_sha256, n_frames=16, stride=45):
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    raw = urllib.request.urlopen(req, timeout=120).read()

    got = hashlib.sha256(raw).hexdigest()
    if got != expected_sha256:
        raise ValueError(
            f"checksum mismatch: expected {expected_sha256}, got {got}"
        )

    frames = []
    source_indices = []

    for i, frame in enumerate(
        iio.imiter(io.BytesIO(raw), plugin="FFMPEG", extension=".webm")
    ):
        if i % stride == 0:
            frames.append(frame)
            source_indices.append(i)
            if len(frames) == n_frames:
                break

    return np.stack(frames), np.asarray(source_indices)

real_video, source_indices = fetch_verified_video(VIDEO_URL, VIDEO_SHA256)

assert real_video.shape == (16, 540, 960, 3), real_video.shape

# Same numerical shape as digit_batch: (8, 8, 8), but different semantics.
r0 = real_video.shape[1] // 2 - 4
c0 = real_video.shape[2] // 2 - 4
video_patch = (
    real_video[:8, r0:r0 + 8, c0:c0 + 8]
    .mean(axis=3)
    .astype(np.float32)
)

print("EN: Setup ready with real image and video data.")
print("ES: Preparación lista con datos reales de imágenes y video.")
print()
print("real_digit / dígito real:", real_digit.shape)
print("digit_batch / lote de dígitos:", digit_batch.shape)
print("real_photo / foto real:", real_photo.shape)
print("real_video / video real:", real_video.shape)
print("video_patch / recorte temporal:", video_patch.shape)

## Shape is not semantics / La forma no es la semántica

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

`digit_batch` and `video_patch` are both <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(8,145,178,.14);border:1px solid rgba(8,145,178,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(8, 8, 8)</span>. One is eight
independent images. The other is eight ordered moments.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">THE WHOLE NOTEBOOK IN ONE LINE · TODO EL CUADERNO EN UNA LÍNEA</div>Every operation here is <b>legal</b> on both and <b>correct</b> on only one.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>digit_batch</code> y <code>video_patch</code> son ambos <code>(8, 8, 8)</code>: uno son ocho imágenes independientes y el otro ocho instantes ordenados. Toda operación de este cuaderno es <b>legal</b> en ambos y <b>correcta</b> en uno solo.</div>

## 2.1 Read real tensors as sentences / Lee tensores reales como frases

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Real objects, not empty arrays.

| Real object / Objeto real | Shape / Forma | Order / Orden | Read it as / Léelo como |
|---|---:|---:|---|
| one digit / un dígito | `(8, 8)` | 2 | height × width / alto × ancho |
| digit batch / lote de dígitos | `(8, 8, 8)` | 3 | examples × height × width / ejemplos × alto × ancho |
| RGB photo / foto RGB | `(512, 512, 3)` | 3 | height × width × colour / alto × ancho × color |
| sampled video / video muestreado | `(16, 540, 960, 3)` | 4 | time × height × width × colour / tiempo × alto × ancho × color |
| padded video batch / lote con padding | `(N, T, H, W, C)` | 5 | examples × time × height × width × colour |

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">NOT A LADDER OF QUALITY · NO ES UNA ESCALA DE CALIDAD</div>A higher order is not better, or cleverer. It is <b>more axes</b>. Nothing else.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un orden mayor no significa «mejor» ni «más inteligente». Significa <b>más ejes</b>, y nada más.</div>

### Axis explorer / Explorador de ejes

Pick a real object. The notebook turns its shape into a sentence.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un objeto real: el cuaderno traduce su forma a una frase.</div>

In [ ]:
#@title 🔍 Axis explorer / Explorador de ejes — run me / ejecútame { display-mode: 'form' }

axis_choice = widgets.Dropdown(
    options=[
        ("One digit / Un dígito", "digit"),
        ("Digit batch / Lote de dígitos", "batch"),
        ("RGB photo / Foto RGB", "photo"),
        ("Real video / Video real", "video"),
    ],
    value="batch",
    description="Object / Objeto:",
    style={"description_width": "120px"},
)

def explain_real_tensor(choice):
    items = {
        "digit": (
            real_digit,
            "(H, W)",
            "height × width",
            "alto × ancho",
        ),
        "batch": (
            digit_batch,
            "(N, H, W)",
            "examples × height × width",
            "ejemplos × alto × ancho",
        ),
        "photo": (
            real_photo,
            "(H, W, C)",
            "height × width × colour",
            "alto × ancho × color",
        ),
        "video": (
            real_video,
            "(T, H, W, C)",
            "time × height × width × colour",
            "tiempo × alto × ancho × color",
        ),
    }

    arr, symbols, en, es = items[choice]

    print("Shape / Forma:", arr.shape)
    print("Order / Orden:", arr.ndim)
    print("Axes / Ejes:", symbols)
    print("EN:", en)
    print("ES:", es)

axis_output = widgets.interactive_output(
    explain_real_tensor,
    {"choice": axis_choice},
)

display(widgets.VBox([axis_choice, axis_output]))

## Exercise 1 — read the real shapes / Ejercicio 1 — lee las formas reales

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

For each real tensor, three steps.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">FOR EVERY TENSOR · PARA CADA TENSOR</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>Print its shape and its order.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>Name every axis.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span>Decide whether reordering axis 0 would preserve or change the meaning.</div></div>

One question settles the third step:

**Is axis 0 a collection of independent examples, or part of the internal
structure of one observation?**

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Para cada tensor: <b>1 ·</b> imprime forma y orden, <b>2 ·</b> nombra cada eje, <b>3 ·</b> decide si reorganizar el eje 0 conserva o cambia el significado.<br><br>La pregunta que lo resuelve: ¿el eje 0 son ejemplos independientes, o parte de la estructura interna de una sola observación?</div>

In [ ]:
# TODO 1 / TAREA 1
#
# Inspect / Inspecciona:
#   real_digit
#   digit_batch
#   real_photo
#   real_video
#
# EN:
# 1. Print .shape and .ndim.
# 2. Name every axis.
# 3. Explain what would happen if axis 0 were reordered.
#
# ES:
# 1. Imprime .shape y .ndim.
# 2. Nombra cada eje.
# 3. Explica qué ocurriría si se reorganizara el eje 0.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

objects = [
    (
        "one real digit / un dígito real",
        real_digit,
        "(H, W)",
        "axis 0 is image height; reordering rows scrambles the image",
        "el eje 0 es el alto; reorganizar filas desordena la imagen",
    ),
    (
        "real digit batch / lote real de dígitos",
        digit_batch,
        "(N, H, W)",
        "axis 0 is independent examples; batch order can change",
        "el eje 0 son ejemplos independientes; el orden del lote puede cambiar",
    ),
    (
        "real RGB photo / foto RGB real",
        real_photo,
        "(H, W, C)",
        "axis 0 is image height; reordering rows scrambles the image",
        "el eje 0 es el alto; reorganizar filas desordena la imagen",
    ),
    (
        "real sampled video / video real muestreado",
        real_video,
        "(T, H, W, C)",
        "axis 0 is time; reordering it changes chronology",
        "el eje 0 es tiempo; reorganizarlo cambia la cronología",
    ),
]

for name, arr, axes, en, es in objects:
    print(name)
    print("  shape/forma:", arr.shape)
    print("  order/orden:", arr.ndim)
    print("  axes/ejes:", axes)
    print("  EN:", en)
    print("  ES:", es)
    print()

<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

The count of axes gives the order. The dataset gives the axes their meaning.

- **batch axis** — independent observations.
- **spatial axis** — position inside one image.
- **time axis** — position in an ordered sequence.
- **channel axis** — different measurements at the same position.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0"><b>eje de lote</b> — observaciones independientes.</li><li style="margin:.35em 0"><b>eje espacial</b> — posición dentro de una imagen.</li><li style="margin:.35em 0"><b>eje temporal</b> — posición en una secuencia ordenada.</li><li style="margin:.35em 0"><b>eje de canales</b> — mediciones distintas en la misma posición.</li></ul></div>

</details>

## 2.2 Same shape, different meaning / Misma forma, distinto significado

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

The most important comparison in this notebook:

<div style="margin:1.4em 0;font:400 15px/2.4 ui-sans-serif,system-ui,sans-serif">
<span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(8,145,178,.14);border:1px solid rgba(8,145,178,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">digit_batch · (8, 8, 8)</span> &nbsp;→&nbsp; <b>(N, H, W)</b><br>
<span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(8,145,178,.14);border:1px solid rgba(8,145,178,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">video_patch · (8, 8, 8)</span> &nbsp;→&nbsp; <b>(T, H, W)</b>
</div>

The first axis has the same **size**. It does not have the same **role**.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">EIGHT CARDS · OCHO TARJETAS</div><div style="margin:.55em 0"><b>Batch</b> — eight separate postcards. Reorder them and you still have the same set.</div><div style="margin:.55em 0"><b>Time</b> — eight frames of an animation. Reorder them and the motion is wrong.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El primer eje tiene el mismo <b>tamaño</b>, no el mismo <b>papel</b>. <b>Lote:</b> ocho postales sueltas; cambiar el orden no cambia el conjunto. <b>Tiempo:</b> ocho fotogramas; cambiar el orden estropea el movimiento.</div>

### Compare the two meanings / Compara los dos significados

Switch between **Batch / Lote** and **Time / Tiempo**. Same numbers, different
reading.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Cambia entre <b>Lote</b> y <b>Tiempo</b>: los mismos números, otra lectura.</div>

In [ ]:
#@title 🔍 Batch vs time / Lote contra tiempo — run me / ejecútame { display-mode: 'form' }

meaning_toggle = widgets.ToggleButtons(
    options=[
        ("Batch / Lote", "batch"),
        ("Time / Tiempo", "time"),
    ],
    value="batch",
    description="Meaning / Significado:",
    style={"description_width": "140px"},
)

def show_same_shape_meaning(kind):
    plt.close("all")

    if kind == "batch":
        fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
        for i, ax in enumerate(axes):
            ax.imshow(digit_batch[i], cmap="gray")
            ax.set_title(f"N={i}")
            ax.axis("off")
        plt.suptitle(
            "Same shape (8,8,8): axis 0 = independent examples / "
            "Misma forma: eje 0 = ejemplos independientes"
        )
        plt.tight_layout()
        plt.show()
        print("EN: reordering N changes presentation order, not the identity of the examples.")
        print("ES: reorganizar N cambia el orden de presentación, no la identidad de los ejemplos.")

    else:
        fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
        for i, ax in enumerate(axes):
            ax.imshow(video_patch[i], cmap="gray")
            ax.set_title(f"T={i}")
            ax.axis("off")
        plt.suptitle(
            "Same shape (8,8,8): axis 0 = ordered time / "
            "Misma forma: eje 0 = tiempo ordenado"
        )
        plt.tight_layout()
        plt.show()
        print("EN: reordering T changes chronology.")
        print("ES: reorganizar T cambia la cronología.")

meaning_output = widgets.interactive_output(
    show_same_shape_meaning,
    {"kind": meaning_toggle},
)

display(widgets.VBox([meaning_toggle, meaning_output]))

## Exercise 2 — shuffle batch vs. shuffle time / Ejercicio 2 — reorganiza lote vs. tiempo

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Apply the **same permutation** to the digit batch and to the video sequence.

For the digits, images and labels must move together. For the video, the frames
are the same and their order is not.

Then measure one number: the **mean change between consecutive sampled frames**.

That is not a measure of video quality. It is evidence that the temporal
relationships moved.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Aplica la <b>misma permutación</b> al lote de dígitos y a la secuencia de video. En los dígitos, imágenes y etiquetas se mueven juntas; en el video quedan los mismos fotogramas en otro orden.<br><br>Después mide el cambio promedio entre fotogramas consecutivos: no es una medida de calidad, es la prueba de que la relación temporal cambió.</div>

In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Create perm = rng.permutation(8).
# 2. Apply it to digit_batch AND digit_labels.
# 3. Apply it to video_patch.
# 4. Print original and shuffled labels.
# 5. Compare the mean absolute change between consecutive sampled video frames.
# 6. Explain why the digit set is still the same but the video chronology is not.
#
# ES:
# 1. Crea perm = rng.permutation(8).
# 2. Aplícala a digit_batch Y digit_labels.
# 3. Aplícala a video_patch.
# 4. Imprime etiquetas originales y reorganizadas.
# 5. Compara el cambio absoluto medio entre fotogramas muestreados consecutivos.
# 6. Explica por qué el conjunto de dígitos sigue siendo el mismo pero la cronología del video no.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

perm = rng.permutation(8)

shuffled_digits = digit_batch[perm]
shuffled_labels = digit_labels[perm]
shuffled_video = video_patch[perm]

same_examples = (
    sorted(
        zip(
            digit_labels.tolist(),
            digit_batch.sum(axis=(1, 2)).round(6).tolist(),
        )
    )
    ==
    sorted(
        zip(
            shuffled_labels.tolist(),
            shuffled_digits.sum(axis=(1, 2)).round(6).tolist(),
        )
    )
)

def mean_consecutive_sampled_change(x):
    x = x.astype(np.float32)
    return float(np.mean(np.abs(x[1:] - x[:-1])))

before = mean_consecutive_sampled_change(video_patch)
after = mean_consecutive_sampled_change(shuffled_video)

print("Permutation / Permutación:", perm.tolist())
print("Original labels / Etiquetas originales:", digit_labels.tolist())
print("Shuffled labels / Etiquetas reorganizadas:", shuffled_labels.tolist())
print("Same labeled examples / Mismos ejemplos etiquetados:", same_examples)
print()

print(f"Video change before / Cambio antes: {before:.3f}")
print(f"Video change after  / Cambio después: {after:.3f}")
print(f"After/before ratio / Razón después/antes: {after / before:.2f}x")
print()

print("EN: the digit examples are the same; only their presentation order changed.")
print("ES: los ejemplos de dígitos son los mismos; solo cambió su orden de presentación.")
print("EN: the video frames are the same, but their measured chronology changed.")
print("ES: los fotogramas son los mismos, pero cambió su cronología medida.")

### See the shuffle / Observa la reorganización

Compare four orders: digits before and after, video before and after.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Compara cuatro órdenes: dígitos antes y después, video antes y después.</div>

In [ ]:
#@title 🔀 Shuffle viewer / Visor de reorganización — run me / ejecútame { display-mode: 'form' }

# One shared shuffle to visualize (TODO 2 steps 1-3). The comparison and the
# explanation of *why* it matters stay in the folded solution above.
perm = rng.permutation(len(digit_batch))
shuffled_digits = digit_batch[perm]
shuffled_labels = digit_labels[perm]
shuffled_video = video_patch[perm]

shuffle_view = widgets.Dropdown(
    options=[
        ("Digits — original / Dígitos — original", "digits_original"),
        ("Digits — shuffled / Dígitos — reorganizados", "digits_shuffled"),
        ("Video — original / Video — original", "video_original"),
        ("Video — shuffled / Video — reorganizado", "video_shuffled"),
    ],
    value="digits_original",
    description="View / Vista:",
    style={"description_width": "100px"},
)

def show_shuffle_view(view):
    plt.close("all")
    fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))

    if view == "digits_original":
        arr = digit_batch
        labels = digit_labels
        title = "Independent examples — original order / Ejemplos independientes — orden original"
        cmap = "gray"
    elif view == "digits_shuffled":
        arr = shuffled_digits
        labels = shuffled_labels
        title = "Independent examples — shuffled order / Ejemplos independientes — orden reorganizado"
        cmap = "gray"
    elif view == "video_original":
        arr = video_patch
        labels = np.arange(8)
        title = "Time sequence — original order / Secuencia temporal — orden original"
        cmap = "gray"
    else:
        arr = shuffled_video
        labels = perm
        title = "Time sequence — shuffled order / Secuencia temporal — orden reorganizado"
        cmap = "gray"

    for i, ax in enumerate(axes):
        ax.imshow(arr[i], cmap=cmap)
        ax.set_title(str(labels[i]))
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

shuffle_output = widgets.interactive_output(
    show_shuffle_view,
    {"view": shuffle_view},
)

display(widgets.VBox([shuffle_view, shuffle_output]))

<details>
<summary><strong>What did the shuffle prove? / ¿Qué demostró la reorganización?</strong></summary>

For the digits, the order of independent examples changed — and every image
kept its label.

For the video, the same measured frames stayed, and the chronology did not.

**An operation can leave `shape` and `dtype` untouched and still change the
scientific meaning.**

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>En los dígitos cambió el orden de ejemplos independientes y cada imagen conservó su etiqueta. En el video quedaron los mismos fotogramas y se alteró la cronología. <b>Una operación puede dejar intactos <code>shape</code> y <code>dtype</code> y aun así cambiar el significado científico.</b></div>

</details>

## 2.3 Real videos have different lengths / Los videos reales tienen longitudes distintas

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

A batch is one rectangular block. Real sequences are not: clip A has 4 frames,
clip B has 7, clip C has 5.

**Padding** is the usual answer.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">PADDING, IN FOUR MOVES · PADDING EN CUATRO PASOS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>Take the longest length — <code>7</code>.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>Copy each real clip into a 7-frame slot.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span>Fill the unused positions with a placeholder.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span>Store a <b>mask</b> saying which positions are real.</div></div>

Three students answer 4, 7 and 5 questions. The spreadsheet demands 7 columns
from everyone. The blank cells are **not answers** — and something has to say
so. That something is the mask.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un lote es un bloque rectangular; las secuencias reales no lo son. El padding crea las posiciones que faltan y la máscara dice cuáles contienen datos reales. Tres estudiantes responden 4, 7 y 5 preguntas: si la hoja exige 7 columnas para todos, las celdas vacías <b>no son respuestas</b>.</div>

## Exercise 3 — build an order-5 batch / Ejercicio 3 — construye un lote de orden 5

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Three non-overlapping segments of the real video: 4 frames, 7 frames, 5 frames.

Predict all five answers before you pad.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">PREDICT FIRST · PREDICE PRIMERO</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>The final shape.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>Which axis is <code>N</code>.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span>Which axis is <code>T</code>.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span>How many <code>(N, T)</code> positions are real.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">5</span>How many are padding.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Tres segmentos del video real — 4, 7 y 5 fotogramas. Antes de ejecutar, predice: la forma final, qué eje es <code>N</code>, qué eje es <code>T</code>, cuántas posiciones <code>(N, T)</code> son reales y cuántas son padding.</div>

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. Spatially subsample real_video with real_video[:, ::4, ::4, :].
# 2. Build three non-overlapping clips with lengths 4, 7, and 5.
# 3. Compute T_max.
# 4. Allocate padded with shape (N, T_max, H, W, C).
# 5. Build a Boolean validity mask with shape (N, T_max).
# 6. Count measured slots and padding slots.
#
# ES:
# 1. Submuestrea espacialmente real_video con real_video[:, ::4, ::4, :].
# 2. Construye tres clips no superpuestos de longitudes 4, 7 y 5.
# 3. Calcula T_max.
# 4. Crea padded con forma (N, T_max, H, W, C).
# 5. Construye una máscara booleana de validez con forma (N, T_max).
# 6. Cuenta posiciones medidas y posiciones de padding.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

video_small = real_video[:, ::4, ::4, :]  # measured pixels, spatially subsampled

real_clips = [
    video_small[0:4],    # 4 measured frames
    video_small[4:11],   # 7 measured frames
    video_small[11:16],  # 5 measured frames
]

lengths = np.array([len(x) for x in real_clips])
T_max = int(lengths.max())
N = len(real_clips)
H, W, C = video_small.shape[1:]

padded = np.zeros((N, T_max, H, W, C), dtype=video_small.dtype)
valid = np.zeros((N, T_max), dtype=bool)

for n, x in enumerate(real_clips):
    T = len(x)
    padded[n, :T] = x
    valid[n, :T] = True

padded_slots = int((~valid).sum())
total_slots = int(valid.size)
measured_slots = int(valid.sum())

print("Clip lengths / Longitudes:", lengths.tolist())
print("Padded shape / Forma con padding:", padded.shape)
print("Order / Orden:", padded.ndim)
print("Axes / Ejes: (N, T, H, W, C)")
print("Validity mask / Máscara de validez:", valid.shape)
print("Measured frame slots / Posiciones medidas:", measured_slots)
print("Padding frame slots / Posiciones de padding:", padded_slots)
print(f"Padding fraction / Fracción de padding: {padded_slots / total_slots:.1%}")

assert padded.shape == (3, 7, 135, 240, 3)
assert valid.sum() == 16

In [ ]:
# The notebook builds the padded order-5 batch here, in a visible cell, so the
# REAL/PAD map and the padding explorer below run whether or not the folded
# solution was executed. The slot-count analysis and the asserts stay folded.
video_small = real_video[:, ::4, ::4, :]

real_clips = [
    video_small[0:4],    # 4 measured frames
    video_small[4:11],   # 7 measured frames
    video_small[11:16],  # 5 measured frames
]

lengths = np.array([len(x) for x in real_clips])
T_max = int(lengths.max())
N = len(real_clips)
H, W, C = video_small.shape[1:]

padded = np.zeros((N, T_max, H, W, C), dtype=video_small.dtype)
valid = np.zeros((N, T_max), dtype=bool)

for n, x in enumerate(real_clips):
    T = len(x)
    padded[n, :T] = x
    valid[n, :T] = True

### REAL vs PAD map / Mapa REAL vs PAD

The map shows the batch at the `(N, T)` level.

- **REAL** — a measured frame exists here.
- **PAD** — none does; the position only keeps the batch rectangular.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El mapa muestra el lote en el nivel <code>(N, T)</code>. <b>REAL</b> = existe un fotograma medido. <b>PAD</b> = no existe; esa posición solo mantiene rectangular el lote.</div>

In [ ]:
#@title 🗺️ REAL vs PAD map / Mapa REAL contra PAD — run me / ejecútame { display-mode: 'form' }

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.imshow(valid, cmap="Greys", vmin=0, vmax=1, aspect="auto")

for n in range(N):
    for t in range(T_max):
        text = "REAL" if valid[n, t] else "PAD"
        text_color = "white" if valid[n, t] else "black"
        ax.text(
            t,
            n,
            text,
            ha="center",
            va="center",
            color=text_color,
            fontsize=9,
            fontweight="bold",
        )

ax.set_xticks(range(T_max))
ax.set_xlabel("Time slot T / Posición temporal T")
ax.set_yticks(range(N))
ax.set_yticklabels(
    [f"clip {n} · measured T={lengths[n]}" for n in range(N)]
)
ax.set_ylabel("Clip N / Video N")
ax.set_title(
    "Measured frames vs padding / Fotogramas medidos vs padding"
)
plt.tight_layout()
plt.show()

### Padding explorer / Explorador de padding

Pick a clip and a time position. A real slot shows its measured frame.

A padded slot shows a **PAD card**, not a black image — because a black image
is exactly what a real measurement of darkness would look like.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">TRY BOTH · PRUEBA LAS DOS</div><div style="margin:.55em 0"><code>Clip N = 0, Time T = 0</code> → REAL</div><div style="margin:.55em 0"><code>Clip N = 2, Time T = 6</code> → PAD</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un clip y una posición. Si es real verás el fotograma medido; si es padding verás una tarjeta <b>PAD</b>, no una imagen negra — una imagen negra es justo lo que parecería una medición real de oscuridad.</div>

In [ ]:
#@title 🔍 Padding explorer / Explorador de padding — run me / ejecútame { display-mode: 'form' }

clip_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=N - 1,
    step=1,
    description="Clip N / Video N:",
    continuous_update=False,
    style={"description_width": "120px"},
)

time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=T_max - 1,
    step=1,
    description="Time T / Tiempo T:",
    continuous_update=False,
    style={"description_width": "120px"},
)

def explore_padding(clip_idx, time_idx):
    slot = padded[clip_idx, time_idx]
    is_valid = bool(valid[clip_idx, time_idx])

    print(
        f"Index / Índice: padded[{clip_idx}, {time_idx}] | "
        f"shape/forma={slot.shape} | valid/válido={is_valid}"
    )

    fig, ax = plt.subplots(figsize=(6.4, 3.6))

    if is_valid:
        ax.imshow(slot)
        ax.set_title("REAL frame / Fotograma REAL")
        ax.axis("off")
        print("EN: a measured frame exists at this position.")
        print("ES: existe un fotograma medido en esta posición.")
    else:
        ax.set_facecolor("#f2f2f2")
        ax.text(
            0.5,
            0.58,
            "PAD",
            ha="center",
            va="center",
            fontsize=34,
            fontweight="bold",
            color="#993333",
            transform=ax.transAxes,
        )
        ax.text(
            0.5,
            0.38,
            "No measured frame exists here\n"
            "No existe un fotograma medido aquí",
            ha="center",
            va="center",
            fontsize=11,
            transform=ax.transAxes,
        )
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(
            "Padding only keeps the batch rectangular / "
            "El padding solo mantiene el lote rectangular"
        )
        print("EN: this slot is padding, not a measured black frame.")
        print("ES: esta posición es padding, no un fotograma negro medido.")

    plt.tight_layout()
    plt.show()

padding_output = widgets.interactive_output(
    explore_padding,
    {
        "clip_idx": clip_slider,
        "time_idx": time_slider,
    },
)

display(
    widgets.VBox([
        widgets.HBox([clip_slider, time_slider]),
        padding_output,
    ])
)

<details>
<summary><strong>Why padding needs a mask / Por qué el padding necesita una máscara</strong></summary>

The padded tensor has a convenient rectangular shape. Not every `(N, T)`
position holds a measured frame.

The mask answers one question: **should the model treat this position as real
data?**

Without it, the zeros of padding are indistinguishable from genuine
observations.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El tensor con padding es rectangular y cómodo, pero no toda posición <code>(N, T)</code> contiene un fotograma medido. La máscara responde una pregunta: <b>¿debe el modelo tratar esta posición como dato real?</b> Sin ella, los ceros del relleno son indistinguibles de observaciones reales.</div>

</details>

## 2.4 The same idea in science / La misma idea en ciencia

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Nothing here is about video. Picture a microscope recording cells over time —
the tensor could be <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(8,145,178,.14);border:1px solid rgba(8,145,178,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(N, T, H, W, C)</span>:

- `N` — patient, dish, well or field of view;
- `T` — measurement time;
- `H`, `W` — image height and width;
- `C` — imaging channels.

The convention depends on the experiment, so it has to be written down.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">A TRAP · UNA TRAMPA</div>A <b>field of view</b> is not automatically <code>H</code> or <code>W</code>. Those two are pixel coordinates <i>inside one image</i>.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La misma lógica vale en ciencia: <code>N</code> puede ser paciente, plato, pozo o campo de visión; <code>T</code> el tiempo de medición; <code>H</code> y <code>W</code> el alto y el ancho; <code>C</code> los canales. La convención depende del experimento y debe documentarse. Un <b>campo de visión</b> no es automáticamente <code>H</code> ni <code>W</code>: esos son coordenadas de píxel dentro de una imagen.</div>

### Experimental-axis explorer / Explorador de ejes experimentales

Pick an axis. Read what it could mean in a microscopy experiment.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un eje y observa qué podría significar en un experimento de microscopía.</div>

In [ ]:
#@title 🔬 Experimental-axis explorer / Explorador de ejes experimentales — run me / ejecútame { display-mode: 'form' }

experiment_axis = widgets.ToggleButtons(
    options=["N", "T", "H", "W", "C"],
    value="N",
    description="Axis / Eje:",
    style={"description_width": "90px"},
)

def explain_experiment_axis(axis):
    explanations = {
        "N": (
            "independent observation: patient, well, dish, or field of view",
            "observación independiente: paciente, pozo, plato o campo de visión",
        ),
        "T": (
            "measurement time or acquisition step",
            "tiempo de medición o paso de adquisición",
        ),
        "H": (
            "pixel rows inside one image",
            "filas de píxeles dentro de una imagen",
        ),
        "W": (
            "pixel columns inside one image",
            "columnas de píxeles dentro de una imagen",
        ),
        "C": (
            "colour, stain, fluorescence, or other measurement channels",
            "canales de color, tinción, fluorescencia u otras mediciones",
        ),
    }

    en, es = explanations[axis]
    print(f"{axis}")
    print("EN:", en)
    print("ES:", es)

experiment_output = widgets.interactive_output(
    explain_experiment_axis,
    {"axis": experiment_axis},
)

display(widgets.VBox([experiment_axis, experiment_output]))

## What just happened / Qué acaba de pasar

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

One rule, learned on real image and video data: **every axis needs a meaning**.

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">FOUR IDEAS · CUATRO IDEAS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span><b>Same shape, different things.</b> <code>(N,H,W)</code> and <code>(T,H,W)</code> can be numerically identical.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span><b>Batch and time are not interchangeable.</b> Reordering examples is not reordering chronology.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span><b>Padding is not measured data.</b> A validity mask says which positions are real.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#0891b2;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span><b>Higher order just means more axes.</b> Not a better model, not a harder one.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una regla, aprendida sobre datos reales: <b>cada eje necesita un significado</b>. <b>1 ·</b> la misma forma puede significar cosas distintas; <b>2 ·</b> lote y tiempo no son intercambiables; <b>3 ·</b> el relleno no son datos medidos; <b>4 ·</b> más orden es solo más ejes.</div>

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

## Done with this section / Fin de esta sección

Next / Siguiente: **03 · Indexing and broadcasting real data / Indexación y broadcasting con datos reales** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/03-indexing-and-broadcasting.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)